<div dir="rtl">

# جيم بوت — تطبيق مقرر TM351 (إدارة البيانات وتحليلها)

هذا الدفتر يمر بدورة حياة البيانات كاملة على بيانات جيم بوت:

1. **جمع البيانات** من قاعدة PostgreSQL ومن السجلات الخام (JSONL)
2. **الأخلاقيات والخصوصية**: الأسماء المستعارة، وفحص k-anonymity
3. **التنظيف والتجهيز** بمكتبة pandas، مع التحقق من النتيجة مقابل قرارات الخادم
4. **التحليل** والرسوم
5. **قاعدة البيانات العلائقية**: استعلامات SQL، وأثر الفهارس
6. **قاعدة بيانات المستندات**: جلسات الروبوت في MongoDB
7. **ما وجدناه وما أصلحناه في التطبيق**

**مصدر البيانات (بصراحة):** البيانات أنتجها محاكي (`simulate_players`) لـ 300 مستخدم خلال 30 يوماً.
الروبوت يلعب فعلاً بمحرك الألعاب نفسه، والمكافآت والحدود من جدول المهام، والجلسات بدون اتصال أُرسلت عبر
مسار المزامنة الحقيقي، فالخادم هو من قبلها أو رفضها. أما **سلوك المستخدمين** (متى يلعبون، وكم مرة، ونسبة اللعب بدون
اتصال، وأخطاء أجهزتهم) فهو افتراضات المحاكي، ولا تُستنتج منه حقائق عن مستخدمين حقيقيين.

</div>

In [1]:
import os, sys, json, warnings
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "analytics").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from scipy import stats
from analytics import clean, analysis as A, mongo, sqlrun, figures as F
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
EXPORT = ROOT / "data" / "export"
FIG = ROOT / "analytics" / "figures"; FIG.mkdir(exist_ok=True)

<div dir="rtl">

## 1. جمع البيانات

البيانات صُدّرت من قاعدة PostgreSQL بأمر Django:

```bash
python manage.py simulate_players --users 300 --days 30 --seed 351
python manage.py export_dataset
```

خمسة جداول CSV (من القاعدة العلائقية) وملفان JSONL (بيانات خام متداخلة):
</div>

In [2]:
t = clean.load_tables(EXPORT)
pd.DataFrame({name: {"rows": len(df), "columns": len(df.columns)} for name, df in t.items()}).T

,rows,columns
users,300,7
platforms,141,5
tasks,11,7
earnings,92730,9
sessions,9061,4


In [3]:
t["earnings"].head()

,earning_id,user,platform_id,task_id,cents,source,offline,client_id,created_at
0,1,u_fc9f3f0eec,1,2.0,50,bot,0,NaN,2026-09-06T17:58:09.350000+00:00
1,2,u_fc9f3f0eec,2,6.0,50,bot,0,NaN,2026-09-06T17:58:09.700000+00:00
2,3,u_fc9f3f0eec,1,1.0,100,bot,0,NaN,2026-09-06T17:58:15.730000+00:00
3,4,u_fc9f3f0eec,1,1.0,100,bot,0,NaN,2026-09-06T17:58:21.040000+00:00
4,5,u_fc9f3f0eec,1,1.0,100,bot,0,NaN,2026-09-06T17:58:26.590000+00:00


In [4]:
t["earnings"].dtypes

earning_id       int64
user               str
platform_id      int64
task_id        float64
cents            int64
source             str
offline          int64
client_id          str
created_at         str
dtype: object

<div dir="rtl">

## 2. الأخلاقيات والخصوصية قبل التحليل

* لا يوجد اسم مستخدم في البيانات المصدّرة: كل مستخدم له **اسم مستعار** = HMAC(المفتاح السري، المعرّف).
* الملاحظات النصية الحرة وتواقيع HMAC حُذفت، وتاريخ التسجيل قُرّب إلى اليوم (**تقليل البيانات**).
* لكن الاسم المستعار وحده لا يكفي: قد يُعرف الشخص من **تركيبة** صفات (معرّفات شبه شخصية).
  نقيس ذلك بـ **k-anonymity**: أصغر مجموعة من المستخدمين يتشاركون نفس القيم.
</div>

In [5]:
quasi = ["joined_date", "daily_limit_min", "text_scale", "dark_mode", "leaderboard", "demo_games_linked"]
before = A.k_anonymity(t["users"], quasi)
gen = A.generalise_users(t["users"])
after = A.k_anonymity(gen, ["joined_week", "daily_limit_min"])
pd.DataFrame([before, after], index=["قبل التعميم", "بعد التعميم"])

,quasi,k,unique_pct,groups
قبل التعميم,"[joined_date, daily_limit_min, text_scale, dar...",1,47.0,209
بعد التعميم,"[joined_week, daily_limit_min]",3,0.0,15


<div dir="rtl">

قبل التعميم كان نحو نصف المستخدمين **متفردين** (k = 1)، أي يمكن تمييزهم لو عرف أحد تاريخ تسجيلهم وإعداداتهم.
بعد استبدال اليوم بالأسبوع وحذف إعدادات المظهر صار k = 3. لذلك أي نشر للبيانات خارج فريق التطوير يجب أن
يستخدم الجدول المعمّم.
</div>

<div dir="rtl">

## 3. التنظيف والتجهيز

### 3.1 جدول الأرباح المرتب (tidy)
ربط الأرباح بالمنصات والمهام (merge)، وتحويل الوقت من UTC إلى توقيت الكويت، واشتقاق التاريخ والساعة واليوم والقناة.
</div>

In [6]:
e = clean.tidy_earnings(t)
e[["user", "platform_key", "task_key", "dollars", "channel", "created_at", "hour", "weekday"]].head()

,user,platform_key,task_key,dollars,channel,created_at,hour,weekday
0,u_fc9f3f0eec,coins,daily,0.5,online,2026-09-06 20:58:09.350000+03:00,20,Sunday
1,u_fc9f3f0eec,guess,daily,0.5,online,2026-09-06 20:58:09.700000+03:00,20,Sunday
2,u_fc9f3f0eec,coins,round,1.0,online,2026-09-06 20:58:15.730000+03:00,20,Sunday
3,u_fc9f3f0eec,coins,round,1.0,online,2026-09-06 20:58:21.040000+03:00,20,Sunday
4,u_fc9f3f0eec,coins,round,1.0,online,2026-09-06 20:58:26.590000+03:00,20,Sunday


<div dir="rtl">

### 3.2 الجلسات المكررة
</div>

In [7]:
sessions, dup_sessions = clean.dedupe_sessions(t["sessions"])
print("جلسات مكررة حُذفت:", dup_sessions, "| بقي:", len(sessions))

جلسات مكررة حُذفت: 81 | بقي: 8980


<div dir="rtl">

السبب: عند إعادة إرسال طلب المزامنة (انقطاع في المنتصف) يسجل الخادم مدة الجلسة مرتين، لأن الجلسات لم يكن
لها مفتاح فريد مثل الأرباح. **أُصلح في الخادم** (القسم 7).

### 3.3 طلبات المزامنة الخام: من JSON متداخل إلى جدول
</div>

In [8]:
raw = clean.load_sync_requests(EXPORT / "raw_sync_requests.jsonl")
print(raw.shape)
raw["ts_format"].value_counts()

(20417, 11)


ts_format
+00:00    12280
Z          6139
+03:00     1998
Name: count, dtype: int64

<div dir="rtl">

ثلاث صيغ مختلفة للوقت من التطبيقات (`Z` و `+00:00` و `+03:00`)، و `pd.to_datetime(utc=True, format="ISO8601")` يوحّدها كلها.

### 3.4 إعادة اشتقاق قرار الخادم لكل سجل، والتحقق منه
</div>

In [9]:
classified = clean.classify_sync(raw, t["earnings"])
check = clean.check_against_server(classified, EXPORT / "raw_sync_requests.jsonl")
print(f"طلبات مطابقة لرد الخادم: {check['match'].sum()} من {len(check)} ({100 * check['match'].mean():.1f}%)")
reasons, by_version = A.sync_quality(classified)
reasons

طلبات مطابقة لرد الخادم: 1482 من 1482 (100.0%)


,records,pct
reason,,
accepted,15480,75.8
daily_cap,3050,14.9
duplicate,760,3.7
bad_task,554,2.7
stale,292,1.4
future,281,1.4


In [10]:
by_version

reason,accepted,bad_task,daily_cap,duplicate,future,stale
app_version,,,,,,
1.0,0.0,100.0,0.0,0.0,0.0,0.0
2.1,77.9,0.0,15.4,3.8,1.4,1.5


<div dir="rtl">

**كل** ردود الخادم (1,482 طلباً) طابقت تصنيفنا، فالتنظيف صحيح. واكتشاف مهم: **كل** سجلات النسخة القديمة 1.0
رُفضت، لأنها ترسل `Round` بحرف كبير. مستخدموها يخسرون كل أرباحهم بدون اتصال. **أُصلح في الخادم.**

### 3.5 توحيد أسماء المنصات التي يكتبها المستخدمون
</div>

In [11]:
names = A.platform_names(t["platforms"], clean.normalise_platform_name)
print("أسماء مختلفة قبل التوحيد:", t["platforms"].query("is_demo == 0")["name"].nunique(), "| بعده:", len(names))
names

أسماء مختلفة قبل التوحيد: 19 | بعده: 5


,platforms,spellings,examples
canonical,,,
Roblox,42,3,ROBLOX | Roblox | roblox
PUBG Mobile,27,5,PUBG | PUBG Mobile | Pubg-Mobile | pubg mobile
Clash of Clans,26,4,COC | Clash Of Clans | Clash of Clans | clash ...
Fortnite,24,3,FORTNITE | Fortnite | fortnite
Free Fire,19,4,FREE FIRE | Free Fire | Free-Fire | freefire


<div dir="rtl">

## 4. التحليل
</div>

In [12]:
pd.Series(A.headline(e, t["users"], sessions))

users                     300.00
earning_records         92730.00
dollars                 94831.00
sessions                 8980.00
days                       31.00
dollars_per_user_day       22.23
dtype: float64

In [13]:
by_game = A.earnings_by_game_task(e)
F.earnings_by_game(by_game, "ar", FIG / "earnings_by_game_ar.png")
by_game

task_key,daily,round,streak,tournament,total
platform_key,,,,,
coins,2129.5,26908.0,6533.0,5494.0,41064.5
guess,2129.5,26908.0,0.0,5494.0,34531.5
xo,982.0,12446.0,2383.0,2592.0,18403.0


In [14]:
A.channel_share(e)

,records,dollars,pct_dollars
channel,,,
online,76970,77742.5,82.0
offline,15480,16256.5,17.1
manual,280,832.0,0.9


<div dir="rtl">

### 4.1 متى يلعب المستخدمون؟
(هذا الرسم يعكس افتراضات المحاكي عن الشخصيات، وفائدته هنا إظهار طريقة التحليل: pivot_table ثم خريطة حرارية.)
</div>

In [15]:
hw = A.hour_weekday(sessions)
F.heatmap(hw, "ar", FIG / "heatmap_ar.png");

<div dir="rtl">

### 4.2 هل يحتاج الروبوت حداً يومياً؟
</div>

In [16]:
per_day, mins = A.minutes_per_day(sessions, t["users"])
F.minutes_hist(per_day, "ar", FIG / "minutes_ar.png")
pd.Series(mins)

user_days         4293.00
mean_min             1.28
p95_min              2.70
max_min              4.27
over_limit_pct       0.00
session_mean_s      36.60
session_max_s       77.00
dtype: float64

<div dir="rtl">

هذه نتيجة حقيقية لأنها تأتي من تأخيرات الخادم نفسها: الجلسة تستغرق نحو 37 ثانية، وأعلى يوم لم يتجاوز 5 دقائق،
فلم يصل أي مستخدم لحده اليومي (أصغر حد 30 دقيقة). **الحد اليومي في التطبيق يقيس وقت الروبوت فقط**،
أما الوقت الذي يقضيه المستخدم في الألعاب نفسها فلا يقيسه، وهذا قيد يجب ذكره بوضوح في شاشة «اللعب الصحي».
</div>

<div dir="rtl">

## 5. قاعدة البيانات العلائقية (PostgreSQL)

الاستعلامات في `analytics/sql/queries.sql`، وتشغيلها مباشرة على القاعدة:
</div>

In [17]:
conn = sqlrun.connect()
Q = sqlrun.load_queries()
list(Q)

['q1_daily_earnings_by_game',
 'q2_top_earners_window',
 'q3_minutes_vs_limit',
 'q4_duplicate_sessions',
 'q5_offline_share_by_hour',
 'q6_custom_platform_names']

In [18]:
sqlrun.run(conn, Q["q1_daily_earnings_by_game"]).tail(6)

,day,game,records,dollars
86,2026-10-04,xo,609,612.00
87,2026-10-05,coins,1559,1562.00
88,2026-10-05,guess,1308,1311.00
89,2026-10-05,xo,687,696.00
90,2026-10-06,coins,7,8.00
91,2026-10-06,guess,6,7.00


In [19]:
sqlrun.run(conn, Q["q2_top_earners_window"])

,id,dollars,rank
0,238,743.00,1
1,83,723.50,2
2,4,694.00,3
3,168,681.00,4
4,255,668.50,5
5,132,626.50,6
6,25,605.00,7
7,29,561.50,8
8,271,560.00,9
9,127,537.50,10


In [20]:
sqlrun.run(conn, Q["q3_minutes_vs_limit"]).head(5)

,user_id,day,minutes,daily_limit_min,over_limit
0,113,2026-09-19,4.3,0,0
1,280,2026-09-20,4.2,120,0
2,295,2026-09-15,4.2,90,0
3,231,2026-09-21,4.1,90,0
4,255,2026-09-09,4.1,90,0


In [21]:
dups_now = sqlrun.run(conn, Q["q4_duplicate_sessions"])
print("جلسات مكررة في القاعدة الآن:", len(dups_now), "(كانت 81 قبل ترحيل الإصلاح 0003)")

جلسات مكررة في القاعدة الآن: 0 (كانت 81 قبل ترحيل الإصلاح 0003)


In [22]:
sqlrun.run(conn, Q["q5_offline_share_by_hour"]).head(8)

,hour,records,offline_pct
0,0,76,28.9
1,6,3185,31.5
2,7,2507,26.1
3,8,2602,34.2
4,9,972,14.4
5,10,1017,7.2
6,11,1170,10.2
7,12,3867,13.5


<div dir="rtl">

### 5.1 أثر الفهارس
نحذف الفهرس، ونقيس زمن التنفيذ بـ `EXPLAIN ANALYZE` (وسيط 15 مرة)، ثم نعيده ونقيس مرة أخرى.
</div>

In [23]:
bench = sqlrun.bench(conn)
F.index_bench(bench, "ar", FIG / "index_ar.png")
pd.DataFrame(bench)[["label", "index", "ms_without", "ms_with", "speedup", "plan_without", "plan_with"]]

,label,index,ms_without,ms_with,speedup,plan_without,plan_with
0,أرباح مستخدم واحد في آخر 7 أيام,"core_earningrecord (user_id, created_at)",0.227,0.064,3.5,Bitmap Heap Scan; Bitmap Index Scan (core_earn...,Bitmap Heap Scan; Bitmap Index Scan (core_earn...
1,أرباح كل المستخدمين في يوم واحد,core_earningrecord (created_at),1.190,0.024,49.6,Index Scan (core_earnin_user_id_5c1350_idx),Index Scan (tm351_earning_created_idx)


<div dir="rtl">

## 6. قاعدة بيانات المستندات (MongoDB)

كل جلسة روبوت مستند واحد فيه مصفوفة الألعاب. إذا ضُبط `MONGO_URL` يُستخدم MongoDB حقيقي، وإلا `mongomock`
(بنفس واجهة pymongo).
</div>

In [24]:
coll = mongo.collection()
print("مستندات:", mongo.load(coll, EXPORT / "raw_bot_sessions.jsonl"))
coll.find_one({}, {"_id": 0, "games": {"$slice": 2}})

مستندات: 7340


{'session_id': '872f38d9-fdbb-d8f3-e98b-948fa8eb0e87',
 'started_at': '2026-09-06T20:58:09+03:00',
 'mode': 'online',
 'rounds_setting': 5,
 'games': [{'game': 'coins',
   'tournament': False,
   'moves': 24,
   'logs': 7,
   'result': 'جمع كل العملات في 24 خطوة',
   'won': True},
  {'game': 'coins',
   'tournament': False,
   'moves': 18,
   'logs': 7,
   'result': 'جمع كل العملات في 18 خطوة',
   'won': True}],
 'earned_cents': 1200,
 'seconds': 47,
 'user': 'u_fc9f3f0eec'}

In [25]:
pd.DataFrame(mongo.per_game_stats(coll)).set_index("_id")

,rounds,avg_moves,max_moves,min_moves,wins
_id,,,,,
coins,24444,21.359025,45,8,24444
guess,24444,0.000000,0,0,24444
xo,11257,0.000000,0,0,10103


<div dir="rtl">

### 6.1 هل يتصرف البحث الثنائي كما تقول النظرية؟
لأي رقم سري بين 1 و 1000، عدد الأرقام التي يجدها البحث الثنائي في k محاولة معروف بالضبط (1، 2، 4 ... 256، ثم 489).
نقارن المرصود في آلاف الجولات بالنظرية باختبار مربع كاي:
</div>

In [26]:
observed = mongo.guess_attempts(coll)
g = A.compare_guess(observed)
p_value = stats.chi2.sf(g.attrs["chi2"], g.attrs["dof"])
F.guess_vs_theory(g, "ar", FIG / "guess_ar.png")
print(f"χ² = {g.attrs['chi2']}, درجات الحرية = {g.attrs['dof']}, p = {p_value:.2f}, n = {g.attrs['n']}")
g

χ² = 9.56, درجات الحرية = 9, p = 0.39, n = 24444


,attempts,theory_pct,observed,observed_pct,expected
0,1,0.1,30,0.12,24.44
1,2,0.2,46,0.19,48.89
2,3,0.4,106,0.43,97.78
3,4,0.8,197,0.81,195.55
4,5,1.6,367,1.50,391.10
5,6,3.2,845,3.46,782.21
6,7,6.4,1547,6.33,1564.42
7,8,12.8,3111,12.73,3128.83
8,9,25.6,6203,25.38,6257.66
9,10,48.9,11992,49.06,11953.12


In [27]:
xo = mongo.xo_outcomes(coll)
print(xo, "| نسبة الخسارة:", 0 if "خسر الروبوت" not in xo else xo["خسر الروبوت"])
print("جلسات فيها بطولة ($elemMatch):", mongo.sessions_with_tournament(coll))
pd.DataFrame(mongo.top_sessions(coll, 5))

{'انتهت المباراة بالتعادل': 1154, 'فاز الروبوت بالمباراة': 10103} | نسبة الخسارة: 0
جلسات فيها بطولة ($elemMatch): 2250


,started_at,earned_cents,seconds,user
0,2026-09-06T20:44:06+03:00,2650,76,u_13311a6a39
1,2026-09-09T19:38:33+03:00,2650,73,u_f5160b602c
2,2026-09-09T22:23:13+03:00,2650,72,u_d628957618
3,2026-09-10T19:49:15+03:00,2650,74,u_d36cffc5ec
4,2026-09-10T21:23:48+03:00,2650,74,u_3290623b38


<div dir="rtl">

## 7. ما وجدناه وما أصلحناه في التطبيق

| ما وجده التحليل | الدليل | الإصلاح في الكود |
|---|---|---|
| جلسات مكررة عند إعادة إرسال المزامنة | 81 جلسة (pandas و SQL) | قيد تفرد (user, started_at) + ترحيل يحذف المكرر + `get_or_create` |
| كل سجلات النسخة القديمة تُرفض | 100% من سجلات v1.0 (`bad_task`) | الخادم يوحّد اسم المهمة إلى أحرف صغيرة |
| 15% من سجلات المزامنة تتجاوز حد المهمة اليومي | معظمها مكافأة يومية أُخذت والتطبيق متصل | توصية: التطبيق ينبه أن المكافأة اليومية قد لا تُحتسب |
| شاشة الأرباح تنفذ 7 استعلامات للأسبوع | عدّ الاستعلامات | استعلام GROUP BY واحد (من 14 إلى 8 استعلامات) |
| الحد اليومي لا يصل له أحد | أعلى يوم أقل من 5 دقائق | توصية: توضيح أن الحد يقيس وقت الروبوت فقط |

</div>

In [28]:
F.sync_reasons(reasons, "ar", FIG / "sync_ar.png")
# حفظ النتائج الرئيسية (للتقرير وللرسوم الإنجليزية)
results = {"headline": A.headline(e, t["users"], sessions), "dup_sessions": dup_sessions,
           "k_before": before, "k_after": after, "reasons": reasons["records"].to_dict(),
           "by_version": by_version.to_dict(), "match": int(check["match"].sum()), "requests": len(check),
           "records": len(raw), "ts_formats": raw["ts_format"].value_counts().to_dict(),
           "names_before": int(t["platforms"].query("is_demo == 0")["name"].nunique()), "names_after": len(names),
           "channel": A.channel_share(e).to_dict(), "minutes": mins, "bench": bench,
           "guess": g.to_dict(orient="list"), "guess_attrs": dict(g.attrs), "p_value": float(p_value), "xo": xo,
           "per_game": mongo.per_game_stats(coll), "tournament_sessions": mongo.sessions_with_tournament(coll),
           "by_game": by_game.to_dict()}
(FIG / "results.json").write_text(json.dumps(results, ensure_ascii=False, indent=1, default=float))
print("الرسوم والنتائج محفوظة في", FIG)

الرسوم والنتائج محفوظة في /home/claude/gamebot-py/analytics/figures
